In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import bootstrap
from tqdm import tqdm
import wandb
import tempfile
import os
import torch as th
from pprint import pprint

tqdm.pandas()

from frag_gnn.utils.misc_utils import scatter_l1normalize, flatten_lol
from frag_gnn.loss import sparse_cosine_distance, sparse_jensen_shannon_divergence

In [2]:
metric_cols = [
    "cos_sim",
    "cos_sim_sqrt",
    "js_sim"
]

In [3]:
data = th.load("/scratch/hdd001/home/adamo/frag-gnn/inference_outputs/msg/benchmark/fragnnet_msg/test.pkl")

In [4]:
pprint(data.keys())

dict_keys(['true_mzs', 'true_logprobs', 'true_batch_idxs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'spec_ce', 'ios_ce', 'oos_ce', 'primary_loss', 'null_formula_prob', 'oos_prob', 'loss', 'cos_sim_0.1', 'jss_sqrt_0.1', 'cos_sim_sqrt_0.01', 'jss_0.01', 'jss_sqrt_0.01', 'jss_hun', 'cos_sim_sqrt_0.1', 'cos_hun', 'cos_sim_0.01', 'jss_0.1'])


In [5]:
num_datapoints = len(data["cos_sim_0.01"])
print(num_datapoints)

9790


In [6]:
# find the missing datapoints
spec_df = pd.read_pickle("../data/proc/ms_gymv2/spec_df.pkl")
# mol_df = pd.read_pickle("../data/proc/ms_gymv2/mol_df.pkl")
split_df = pd.read_csv("../data/split/ms_gym_d3_h4_benchmarkv2/test_ids.csv")

In [7]:
print(spec_df.columns)

Index(['spec_id', 'mol_id', 'prec_type', 'inst_type', 'frag_mode', 'spec_type',
       'ion_mode', 'dset', 'dset_spec_id', 'col_gas', 'res', 'ace', 'nce',
       'prec_mz', 'peaks', 'ri', 'group_id'],
      dtype='object')


In [8]:
print(split_df.columns)

Index(['spec_id', 'mol_id', 'group_id'], dtype='object')


In [9]:
test_ids = split_df["spec_id"].values
test_dset_ids = spec_df[spec_df["spec_id"].isin(test_ids)]["dset_spec_id"].values
print(len(test_dset_ids))
print(len(set(test_dset_ids)))
test_dset_ids = pd.Series(test_dset_ids)
# test_dset_ids.to_csv("/h/adamo/MassSpecGym_copy/data/test_dset_ids.csv", index=False)

9790
9790


In [10]:
msg_df = pd.read_csv("/h/adamo/MassSpecGym_copy/data/MassSpecGym.tsv", sep="\t")
msg_df

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge
0,MassSpecGymID0000001,"91.0542,125.0233,154.0499,155.0577,185.0961,20...","0.24524524524524524,1.0,0.08008008008008008,0....",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,30.0,train,True
1,MassSpecGymID0000002,"91.0542,125.0233,155.0577,185.0961,229.0859,24...","0.0990990990990991,0.28128128128128127,0.04004...",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,20.0,train,True
2,MassSpecGymID0000003,"69.0343,91.0542,125.0233,127.039,153.0699,154....","0.03403403403403404,0.31431431431431434,1.0,0....",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,40.0,train,True
3,MassSpecGymID0000004,"69.0343,91.0542,110.06,111.0441,112.0393,120.0...","0.17917917917917917,0.47347347347347346,0.0380...",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,55.0,train,True
4,MassSpecGymID0000005,"91.0542,125.0233,185.0961,229.0859,246.1125,28...","0.07807807807807808,0.1841841841841842,0.03503...",CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,VFMQMACUYWGDOJ,C16H17NO4,C16H18NO4,287.115224,288.1225,[M+H]+,Orbitrap,10.0,train,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231099,MassSpecGymID0414168,"55.054474,58.029369,58.065601,59.049339,69.032...","0.0004935684823754727,0.00012405427491363124,0...",CC[C@@H]1[C@H](/C=C(/C=C\C(=O)[C@@H](C[C@@H]([...,WBPYTXDJUQJLPQ,C46H77NO17,C46H78NO17,915.521724,916.5290,[M+H]+,QTOF,NaN,val,False
231100,MassSpecGymID0414171,"72.043739,86.060242,98.058304,99.0634,126.0537...","0.0019527744563354998,0.0016939927337585416,0....",C[C@@]1([C@H]2C[C@H]3[C@@H](C(=O)C(=C([C@]3(C(...,DHPRQBPJLMKORJ,C22H23ClN2O8,C22H24ClN2O8,478.115724,479.1230,[M+H]+,QTOF,NaN,test,False
231101,MassSpecGymID0414172,"72.080132,102.233917,113.082344,113.121498,114...","0.021173783463613503,0.007516001055215398,0.03...",C[C@H]([C@@H]1CC[C@H]([C@H](O1)O[C@@H]2[C@H](C...,CEAZRRDELHUEMR,C21H43N5O7,C21H44N5O7,477.317724,478.3250,[M+H]+,QTOF,NaN,val,False
231102,MassSpecGymID0414173,"55.053627,56.455425,58.065819,67.053543,82.065...","0.031731527156456024,0.017878400151027027,0.07...",C[C@H]([C@@H]1CC[C@H]([C@H](O1)O[C@@H]2[C@H](C...,CEAZRRDELHUEMR,C21H43N5O7,C21H44N5O7,477.317724,478.3250,[M+H]+,QTOF,NaN,val,False


In [11]:
msg_df["fold"].value_counts()

fold
train    194119
val       19429
test      17556
Name: count, dtype: int64

In [12]:
msg_test_dset_ids = msg_df[msg_df["simulation_challenge"] & (msg_df["fold"] == "test")]["identifier"]
print(len(msg_test_dset_ids))

9954


In [13]:
missing_test_dset_ids = set(msg_test_dset_ids) - set(test_dset_ids)
num_missing_datapoints = len(missing_test_dset_ids)
print(num_missing_datapoints)

164


In [14]:
prec_mzs = msg_df[msg_df["identifier"].isin(missing_test_dset_ids)]["precursor_mz"]
print(len(prec_mzs))
mzs = msg_df[msg_df["identifier"].isin(missing_test_dset_ids)]["mzs"]
ints = msg_df[msg_df["identifier"].isin(missing_test_dset_ids)]["intensities"]
counts = mzs.apply(len)


164


In [15]:
# create target spectra
true_mzs = []
true_ints = []
counts = []
for i in range(len(prec_mzs)):
    mzs_str = mzs.iloc[i]
    ints_str = ints.iloc[i]
    mzs_float = [float(mz) for mz in mzs_str.split(",")]
    ints_float = [float(intensity) for intensity in ints_str.split(",")]
    counts.append(len(mzs_float))
    true_mzs.extend(mzs_float)
    true_ints.extend(ints_float)
true_mzs = th.tensor(true_mzs)
true_ints = th.tensor(true_ints)
counts = th.tensor(counts)
true_batch_idxs = th.repeat_interleave(th.arange(len(prec_mzs)), counts, dim=0)
true_ints = scatter_l1normalize(true_ints, true_batch_idxs)
true_sqrt_ints = th.sqrt(true_ints*1000)
true_sqrt_ints = scatter_l1normalize(true_sqrt_ints, true_batch_idxs)

In [16]:
# create predicted spectra
pred_mzs = th.tensor(prec_mzs.tolist())
pred_ints = th.ones_like(pred_mzs)
pred_batch_idxs = th.arange(len(prec_mzs))

In [17]:
# calculate similarities
cos_sim = 1.-sparse_cosine_distance(
    true_mzs, 
    true_ints,
    true_batch_idxs,
    pred_mzs, 
    pred_ints, 
    pred_batch_idxs,
    mz_max=1005.,
    mz_bin_res=0.01
)
cos_sim_sqrt = 1.-sparse_cosine_distance(
    true_mzs, 
    true_sqrt_ints,
    true_batch_idxs,
    pred_mzs, 
    pred_ints, 
    pred_batch_idxs,
    mz_max=1005.,
    mz_bin_res=0.01
)
js_sim = 1.-sparse_jensen_shannon_divergence(
    true_mzs, 
    true_ints,
    true_batch_idxs,
    pred_mzs, 
    pred_ints, 
    pred_batch_idxs,
    mz_max=1005.,
    mz_bin_res=0.01
)
print(cos_sim.mean())
print(cos_sim_sqrt.mean())
print(js_sim.mean())

tensor(0.0892)
tensor(0.0782)
tensor(0.3524)


In [18]:
total_num_datapoints = num_datapoints + num_missing_datapoints
df = {}
df["cos_sim"] = np.concatenate([data["cos_sim_0.01"].numpy(), cos_sim.numpy()])
df["js_sim"] = np.concatenate([data["jss_0.01"].numpy(), js_sim.numpy()])
df["cos_sim_sqrt"] = np.concatenate([data["cos_sim_sqrt_0.01"].numpy(), cos_sim_sqrt.numpy()])
df["method"] = np.array(["fragnnet" for _ in range(total_num_datapoints)])
df = pd.DataFrame(df)
df

,cos_sim,js_sim,cos_sim_sqrt,method
0,0.948707,0.881730,0.850913,fragnnet
1,0.992814,0.924094,0.893146,fragnnet
2,0.848841,0.838364,0.802144,fragnnet
3,0.311855,0.628174,0.549550,fragnnet
4,0.744671,0.694117,0.589568,fragnnet
...,...,...,...,...
9949,0.000000,0.306853,0.000000,fragnnet
9950,0.000000,0.306853,0.000000,fragnnet
9951,0.000000,0.306853,0.000000,fragnnet
9952,0.000000,0.306853,0.000000,fragnnet


In [19]:
def calculate_ci(df, metric_cols):

    # Calculate means for all metrics into a single table
    df[metric_cols] = df[metric_cols]*100
    df_mean = df.groupby('method')[metric_cols].mean().round(2)

    # Calculate confidence intervals for all metrics into a single table
    def get_ci(col_vals, confidence_level=0.999, n_resamples=20_000):
        res = bootstrap((col_vals,), np.mean, confidence_level=confidence_level, n_resamples=n_resamples)
        ci = res.confidence_interval
        return f'{ci.low:.2f}-{ci.high:.2f}'
    def get_ci_for_each_col(df_method):
        return df_method.apply(get_ci, axis=0)
    df_ci = df.groupby('method')[metric_cols].progress_apply(lambda df_method: get_ci_for_each_col(df_method))

    # Merge tables with means and confidence intervals
    for col in metric_cols:
        df_mean[col] = df_mean[col].astype(str) + ' (' + df_ci[col] + ')'
    return df_mean

In [20]:
df_ci = calculate_ci(df, metric_cols)
df_ci

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:15<00:00, 15.61s/it]


,cos_sim,cos_sim_sqrt,js_sim
method,,,
fragnnet,51.94 (50.90-53.02),50.83 (50.07-51.58),63.28 (62.75-63.81)


In [21]:
retform_df = pd.read_csv("../msg_retrieval/retrieval_candidates_formula_summary.csv")
retmass_df = pd.read_csv("../msg_retrieval/retrieval_candidates_mass_summary.csv")

In [22]:
# convert ranks to hit rates
retform_df["method"] = "retform_fragnnet"
retform_df["hit_rate@1"] = (retform_df["rank"] == 1).astype(int)
retform_df["hit_rate@5"] = (retform_df["rank"] <= 5).astype(int)
retform_df["hit_rate@20"] = (retform_df["rank"] <= 20).astype(int)
retmass_df["method"] = "retmass_fragnnet"
retmass_df["hit_rate@1"] = (retmass_df["rank"] == 1).astype(int)
retmass_df["hit_rate@5"] = (retmass_df["rank"] <= 5).astype(int)
retmass_df["hit_rate@20"] = (retmass_df["rank"] <= 20).astype(int)
ret_df = pd.concat([retform_df, retmass_df])

In [23]:
ret_df_ci = calculate_ci(ret_df, [f"hit_rate@{k}" for k in [1, 5, 20]])
ret_df_ci

100%|██████████| 2/2 [00:34<00:00, 17.31s/it]


,hit_rate@1,hit_rate@5,hit_rate@20
method,,,
retform_fragnnet,31.93 (30.40-33.50),63.2 (61.64-64.76),82.7 (81.45-83.93)
retmass_fragnnet,46.64 (44.98-48.26),72.56 (71.18-74.00),83.58 (82.34-84.75)
